# Data Processing Pipeline

This notebook applies a processing pipeline to raw borehole data (Conductivity vs. Depth). Each step of the pipeline is executed sequentially, and a comparative plot is generated to visualize the effect of the transformation.

The processing functions are imported from the `01_processing.py` script located in the `../modules/` directory.

In [1]:
# === 1. Setup and Imports ===
# This cell imports necessary libraries and the custom processing functions.

import sys
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

# -- Setup project root path to allow imports from other directories --
try:
    root = os.path.abspath('..')
    if root not in sys.path:
        sys.path.append(root)
except NameError:
    # If running in an environment where '..' is not defined, use current dir
    root = os.getcwd()

print(f"Project root set to: {root}")

# -- Import custom processing functions from the module --
try:
    import importlib.util
    # The path to the module is relative to this notebook's location (notebooks/)
    spec = importlib.util.spec_from_file_location("processing_module", "../modules/01_processing.py")
    processing_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(processing_module)

    # Assign functions to variables for easier access
    process_borehole_data = processing_module.process_borehole_data
    DEFAULT_COLUMN_MAPPINGS = processing_module.DEFAULT_COLUMN_MAPPINGS
    adjust_vertical_position = processing_module.adjust_vertical_position
    filter_non_negative_values = processing_module.filter_non_negative_values
    find_column_name = processing_module.find_column_name
    average_grouped_by_depth = processing_module.average_grouped_by_depth
    resample_profile_uniform = processing_module.resample_profile_uniform
    apply_savgol_filter_to_df = processing_module.apply_savgol_filter_to_df
    
    print("Successfully imported processing functions from '01_processing.py'.")

except (ImportError, FileNotFoundError) as e:
    print(f"Error importing processing module: {e}")
    print("Please ensure 'modules/01_processing.py' exists and is accessible from the project root.")

Project root set to: c:\Users\Arhui\Desktop\proyectos\mar\freshwater_lens
Successfully imported processing functions from '01_processing.py'.


## Plotting Helper Function

To keep the code clean and avoid repetition, we define a helper function to create the comparison plots.

In [2]:
def plot_comparison(df_before, df_after, name_before, name_after, depth_col, cond_col, title):
    """
    Generates an aesthetic Plotly graph to compare data before and after a processing step.
    
    Args:
        df_before (pd.DataFrame): DataFrame before the processing step.
        df_after (pd.DataFrame): DataFrame after the processing step.
        name_before (str): Label for the 'before' trace.
        name_after (str): Label for the 'after' trace.
        depth_col (str): Name of the depth column.
        cond_col (str): Name of the conductivity column.
        title (str): Title of the plot.
    """
    fig = go.Figure()

    # Add trace for data BEFORE processing
    fig.add_trace(go.Scatter(
        x=df_before[depth_col],
        y=df_before[cond_col],
        mode='lines',
        name=f"{name_before} (n = {len(df_before)})",
        line=dict(color='royalblue', width=2),
        opacity=0.8
    ))

    # Add trace for data AFTER processing
    fig.add_trace(go.Scatter(
        x=df_after[depth_col],
        y=df_after[cond_col],
        mode='lines',
        name=f"{name_after} (n = {len(df_after)})",
        line=dict(color='firebrick', width=2.5)
    ))

    # Update layout for a clean, professional look
    fig.update_layout(
        title=dict(text=f'<b>{title}</b>', x=0.5, font=dict(size=20)),
        xaxis_title=f"<b>{depth_col}</b>",
        yaxis_title=f"<b>{cond_col}</b>",
        xaxis=dict(gridcolor='lightgrey'),
        yaxis=dict(gridcolor='lightgrey'),
        template='plotly_white',
        legend=dict(
            x=0.01,
            y=0.99,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='black',
            borderwidth=1
        ),
        height=500
    )
    
    fig.show()

## Step 0: Load Raw Data

First, we load the raw CSV file into a pandas DataFrame. We'll use a sample file name, but you can change `file_name` to process a different file. We also auto-detect the correct column names for depth and conductivity using the mappings from our module.

In [3]:
# === Define file and load data ===
file_name = 'AW6_D_YSI_20250226.csv' # <-- CHANGE THIS TO YOUR FILENAME
file_path = os.path.join(root, 'data', 'raw', 'raw_new', file_name)

try:
    df_raw = pd.read_csv(file_path, encoding='utf-8')
    print(f"Successfully loaded '{file_name}' with {len(df_raw)} records.")
    display(df_raw.head())
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'")
    print("Please ensure the file exists and the path is correct.")
    # Create a dummy dataframe to allow the rest of the notebook to run without errors
    df_raw = pd.DataFrame({
        'Vertical Position [m]': [ -1, 0, 1, 2, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'Corrected sp Cond [uS/cm]': [100, 200, 500, 1000, 1100, 1500, 2500, 4000, 6000, 8000, 10000, 12000, 15000]
    })
    print("\nCreated a dummy DataFrame to demonstrate the pipeline.")

# -- Auto-detect column names --
depth_col = find_column_name(df_raw, 'depth', DEFAULT_COLUMN_MAPPINGS)
cond_col = find_column_name(df_raw, 'conductivity', DEFAULT_COLUMN_MAPPINGS)

if depth_col and cond_col:
    print(f"\nDetected Depth Column: '{depth_col}'")
    print(f"Detected Conductivity Column: '{cond_col}'")
else:
    print("\nError: Could not detect depth and/or conductivity columns. Please check column names in the file.")

Successfully loaded 'AW6_D_YSI_20250226.csv' with 11612 records.


,Date (MM/DD/YYYY),Time (HH:MM:SS),Time (Fract. Sec),Site Name,Cond_muS/cm,Depth m,ODO % sat,ODO mg/L,ORP mV,Pressure psi a,Sal psu,SpCond_muS/cm,TDS mg/L,pH,pH mV,Temp_Celcius,Vertical Position m,Battery V,Cable Pwr V,Resistivity ohms-cm
0,26/02/2025,10:15:07,0,Default Site,491.4,0.112,61.6,5.09,82.0,0.158,0.24,491.9,320,7.57,-67.0,24.942,0.114,2.41,1.1,2035
1,26/02/2025,10:15:08,0,Default Site,497.2,0.112,57.2,4.73,82.2,0.159,0.24,497.7,324,7.55,-66.0,24.950,0.113,2.42,1.1,2011
2,26/02/2025,10:15:09,0,Default Site,499.1,0.112,54.5,4.50,82.4,0.159,0.24,499.5,325,7.54,-65.2,24.962,0.113,2.42,1.1,2004
3,26/02/2025,10:15:10,0,Default Site,499.7,0.112,53.0,4.37,82.5,0.159,0.24,500.0,325,7.52,-64.6,24.971,0.112,2.42,1.1,2001
4,26/02/2025,10:15:11,0,Default Site,500.1,0.112,47.4,3.92,82.6,0.159,0.24,500.3,325,7.51,-64.0,24.978,0.113,2.42,1.1,2000



Detected Depth Column: 'Vertical Position m'
Detected Conductivity Column: 'SpCond_muS/cm'


## Step 1: Adjust Vertical Position

The first processing step is to apply a vertical adjustment to the depth measurements. We use the `adjust_vertical_position` function. Here, we'll use the `'TOM'` method as an example.

In [4]:
if 'df_raw' in locals() and depth_col and cond_col:
    # Apply the adjustment
    df_adjusted = adjust_vertical_position(
        df_raw,
        depth_col=depth_col,
        method='TOM', # or 'YSI'
        adjustment=0.272
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_raw,
        df_after=df_adjusted,
        name_before='Raw Data',
        name_after='Adjusted Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 1: Vertical Position Adjustment ({file_name})'
    )

## Step 2: Filter Negative Values

Next, we remove any records with negative depth values, which are physically meaningless for this type of measurement. We apply this to the already adjusted data.

In [5]:
if 'df_adjusted' in locals() and depth_col and cond_col:
    # Apply the filter
    df_positive = filter_non_negative_values(
        df_adjusted, 
        depth_col=depth_col
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_adjusted, 
        df_after=df_positive,
        name_before='Adjusted Data',
        name_after='Positive Depth Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 2: Removing Negative Depth Values ({file_name})'
    )

## Step 3: Average Duplicates by Depth

Some logging tools may record multiple conductivity values at the exact same depth. This step groups the data by each unique depth value and calculates the mean conductivity, ensuring one value per depth.

In [6]:
if 'df_positive' in locals() and depth_col and cond_col:
    # Apply the averaging function
    df_averaged, duplicates = average_grouped_by_depth(
        df_positive,
        depth_col=depth_col,
        value_col=cond_col
    )
    
    print(f"Found and averaged {len(duplicates)} depth points with duplicate records.")
    if not duplicates.empty:
        display(duplicates.head())

    # Plot the comparison
    plot_comparison(
        df_before=df_positive,
        df_after=df_averaged,
        name_before='Positive Depth Data',
        name_after='Averaged Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 3: Averaging Duplicate Depths ({file_name})'
    )

Found and averaged 386 depth points with duplicate records.


,Duplicated Depth,Frequency
1,0.382,2
2,0.383,41
3,0.384,231
4,0.385,290
5,0.386,55


## Step 4: Resample to Uniform Grid

The data is often recorded at irregular depth intervals. To standardize the profile, we resample it to a uniform grid using PCHIP interpolation. The spacing `dz` can be set manually or calculated automatically.

In [7]:
if 'df_averaged' in locals() and depth_col and cond_col:
    # Apply resampling
    df_resampled = resample_profile_uniform(
        df_averaged, 
        depth_col=depth_col, 
        value_col=cond_col, 
        dz_method='median' # Options: 'percentile95', 'median', 'mean', 'min'
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_averaged, 
        df_after=df_resampled,
        name_before='Averaged Data',
        name_after='Resampled Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 4: Resampling to Uniform Grid ({file_name})'
    )

## Step 5 (Optional): Smoothing with Savitzky-Golay Filter

Finally, we can apply a Savitzky-Golay filter to smooth out high-frequency noise in the resampled data, revealing the underlying trend more clearly.

In [8]:
if 'df_resampled' in locals() and depth_col and cond_col:
    # Apply the filter
    df_smoothed = apply_savgol_filter_to_df(
        df_resampled,
        value_col=cond_col,
        window_length=15, # Must be odd, will be adjusted if even
        poly_order=3
    )

    # Plot the comparison
    plot_comparison(
        df_before=df_resampled,
        df_after=df_smoothed,
        name_before='Resampled Data',
        name_after='Smoothed Data',
        depth_col=depth_col,
        cond_col=cond_col,
        title=f'Step 5: Savitzky-Golay Smoothing ({file_name})'
    )
    
    print("\nFinal Processed Data Head:")
    display(df_smoothed.head())


Final Processed Data Head:


,Vertical Position m,SpCond_muS/cm
0,0.381,504.413039
1,0.395,503.785098
2,0.409,503.332157
3,0.423,503.030196
4,0.437,502.855196
